# V19 · Datenbanken und SQL: sqlite3, Platzhalter und SQL-Injection — Vertiefung zum Training

**Wo passt das Notebook hin?** Erst die Python-Werkstatt im Labor, dann das Training zu Hause (Browser, ca. 75 Min). Dieses Notebook ist die **Vertiefung** und läuft in **Codespace**, Jupyter oder Colab. Es nutzt Python aus V01 bis V19: Funktionen, Listen, Tupel, Dictionaries, Dateien, try/except, und neu `sqlite3` mit SQL.

**Der rote Faden:** die Fertigungszelle **FZ-3** in Halle 2 (Drehmaschine D2, Fräsmaschine F1, Roboter R1, Messmaschine M1, Zellensteuerung SPS). In V16 hast du ihre Logs durchsucht, in V17 die Maschinendaten als JSON geholt, in V18 die Firmware von R1 geprüft. Heute bekommt die Zelle eine **Auftragsdatenbank**: wer fertigt was, wie viel, mit welchem Status.

**Keine Datei nötig:** Das Notebook baut die Datenbank selbst im Arbeitsspeicher auf (`:memory:`, dieselben zwölf Aufträge wie in `fz3.db` aus dem Labor). Du musst nichts herunterladen. Jede Zelle, die ändert, arbeitet mit einer frischen Kopie: `db = neue_db()`.

**Der Arbeitsablauf bei jeder Aufgabe:** (1) Die Daten ansehen (`SELECT *`). (2) Die Abfrage schreiben, **Werte immer als `?` mit Tupel**, nie per f-String in den SQL-Text. (3) Bei Änderungen `commit()` nicht vergessen. (4) Die Prüfzelle laufen lassen. Die Prüfzellen sind keine Musterlösungen, sie zeigen nur, ob dein Ergebnis stimmt.

**Stufen:** Aufwärmen (1.x), Anwenden (2.x), Knobeln (3.x). Tipps und Musterlösungen sind eingeklappt: erst selbst versuchen, dann vergleichen.

In [ ]:
# Werkzeug-Zelle: einmal ausführen, nicht verändern.
import sqlite3


def pruefe(ist, soll, text):
    if ist == soll:
        print("OK: " + text)
    else:
        print("FEHLER: " + text + " (erwartet " + str(soll) + ", bei dir steht " + str(ist) + ")")


def pruefe_ja(bedingung, text):
    if bedingung:
        print("OK: " + text)
    else:
        print("FEHLER: " + text)


MASCHINEN = [
    ("D2", "Drehmaschine", "192.168.30.11"),
    ("F1", "Fraesmaschine", "192.168.30.12"),
    ("R1", "Roboter", "192.168.31.20"),
    ("M1", "Messmaschine", "192.168.30.14"),
]

AUFTRAEGE = [
    (4711, "WELLE_20H6", "D2", 500, "in Arbeit"),
    (4712, "FLANSCH_80", "F1", 120, "offen"),
    (4713, "WELLE_20H6", "D2", 300, "offen"),
    (4714, "BUCHSE_30", "D2", 800, "fertig"),
    (4715, "FLANSCH_80", "F1", 60, "fertig"),
    (4716, "DECKEL_80", "F1", 200, "offen"),
    (4717, "BOLZEN_12", "D2", 1500, "offen"),
    (4718, "FLANSCH_80", "F1", 240, "in Arbeit"),
    (4719, "WELLE_25K6", "D2", 150, "offen"),
    (4720, "BUCHSE_30", "D2", 400, "offen"),
    (4721, "DECKEL_80", "F1", 90, "fertig"),
    (4722, "WELLE_20H6", "D2", 250, "fertig"),
]


def neue_db():
    """Baut die Auftragsdatenbank der Zelle FZ-3 frisch im Arbeitsspeicher auf (wie fz3.db im Labor)."""
    db = sqlite3.connect(":memory:")
    cur = db.cursor()
    cur.execute("CREATE TABLE maschinen (id TEXT PRIMARY KEY, typ TEXT, ip TEXT)")
    cur.execute("""CREATE TABLE auftraege (
        nr INTEGER PRIMARY KEY,
        teil TEXT,
        maschine TEXT,
        menge INTEGER,
        status TEXT
    )""")
    cur.executemany("INSERT INTO maschinen VALUES (?, ?, ?)", MASCHINEN)
    cur.executemany("INSERT INTO auftraege VALUES (?, ?, ?, ?, ?)", AUFTRAEGE)
    db.commit()
    return db


db = neue_db()
cur = db.cursor()
cur.execute("SELECT COUNT(*) FROM auftraege")
print("Bereit. Aufträge in der Datenbank:", cur.fetchone()[0])

## Das vorgerechnete Beispiel
Das ist dasselbe Programm wie im Labor (Station „Vorgerechnetes Beispiel“). Lies es Zeile für Zeile und führe die Zelle aus: Es legt eine Tabelle an, fragt die offenen Aufträge von D2 ab, summiert die Menge je Maschine und zeigt, warum ein f-String im SQL-Text gefährlich ist. Erwartete Ausgabe:

```
Auftrag 4711: 500 x WELLE_20H6
Auftrag 4713: 300 x WELLE_20H6
Menge je Maschine: [('D2', 1600), ('F1', 180)]
Unsicher (f-String): 5 Treffer - alle Aufträge!
Sicher (Platzhalter): 0 Treffer
```

In [ ]:
# Fertigungszelle FZ-3: Auftragsdatenbank mit SQL abfragen
# (V19: sqlite3, SQL CREATE/INSERT/SELECT, Parameter-Platzhalter, SQL-Injection)

import sqlite3

db = sqlite3.connect(":memory:")          # Datenbank im Arbeitsspeicher (zum Üben)
cur = db.cursor()
cur.execute("""CREATE TABLE auftraege (
    nr INTEGER PRIMARY KEY,
    teil TEXT,
    maschine TEXT,
    menge INTEGER,
    status TEXT
)""")

auftraege = [
    (4711, "WELLE_20H6", "D2", 500, "in Arbeit"),
    (4712, "FLANSCH_80", "F1", 120, "offen"),
    (4713, "WELLE_20H6", "D2", 300, "offen"),
    (4714, "BUCHSE_30", "D2", 800, "fertig"),
    (4715, "FLANSCH_80", "F1", 60, "fertig"),
]
cur.executemany("INSERT INTO auftraege VALUES (?, ?, ?, ?, ?)", auftraege)
db.commit()

# Alle offenen Aufträge der Drehmaschine D2 - mit Platzhaltern
maschine = "D2"
cur.execute("SELECT nr, teil, menge FROM auftraege WHERE maschine = ? AND status != ?",
            (maschine, "fertig"))
for nr, teil, menge in cur.fetchall():
    print(f"Auftrag {nr}: {menge} x {teil}")

# Summe der Stückzahlen je Maschine
cur.execute("SELECT maschine, SUM(menge) FROM auftraege GROUP BY maschine")
print("Menge je Maschine:", cur.fetchall())

# SQL-Injection: Eingabe direkt in den SQL-Text einbauen ist gefährlich
eingabe = "x' OR '1'='1"
sql = f"SELECT nr FROM auftraege WHERE teil = '{eingabe}'"
cur.execute(sql)
print("Unsicher (f-String):", len(cur.fetchall()), "Treffer - alle Aufträge!")

cur.execute("SELECT nr FROM auftraege WHERE teil = ?", (eingabe,))
print("Sicher (Platzhalter):", len(cur.fetchall()), "Treffer")
db.close()

## Werkzeugkasten in Zellen
Fünf kurze Experimente zum Ändern und Beobachten. Jedes arbeitet mit einer frischen Datenbank.

**1. Was liefert fetchall, was fetchone?** `fetchall()` gibt immer eine Liste von Tupeln (notfalls eine leere), `fetchone()` ein Tupel oder `None`.

In [ ]:
db = neue_db()
cur = db.cursor()

cur.execute("SELECT nr, teil FROM auftraege WHERE maschine = ?", ("F1",))
print(cur.fetchall())                                   # Liste von Tupeln

cur.execute("SELECT nr, teil FROM auftraege WHERE nr = ?", (4711,))
print(cur.fetchone())                                   # ein Tupel

cur.execute("SELECT nr, teil FROM auftraege WHERE nr = ?", (9999,))
print(cur.fetchone())                                   # None: gibt es nicht
print(cur.fetchall())                                   # leere Liste

**2. Das Tupel mit einem Element braucht das Komma.** `("D2",)` ist ein Tupel, `("D2")` nur der Text „D2“ mit zwei Zeichen. Die Fehlermeldung von SQLite zählt die Zeichen:

In [ ]:
db = neue_db()
cur = db.cursor()
maschine = "D2"

cur.execute("SELECT COUNT(*) FROM auftraege WHERE maschine = ?", (maschine,))
print("mit Komma:", cur.fetchone())

try:
    cur.execute("SELECT COUNT(*) FROM auftraege WHERE maschine = ?", (maschine))
except sqlite3.ProgrammingError as fehler:
    print("ohne Komma:", fehler)

**3. Ohne commit ist die Änderung weg.** Dieses Experiment braucht eine Datei, denn bei `:memory:` ist ohnehin alles beim Schließen weg. Es legt eine temporäre Datei an, ändert einen Auftrag einmal ohne und einmal mit `commit()` und liest mit einer **zweiten Verbindung** nach.

In [ ]:
import os
import tempfile

pfad = os.path.join(tempfile.mkdtemp(), "test.db")
db = sqlite3.connect(pfad)
db.execute("CREATE TABLE auftraege (nr INTEGER PRIMARY KEY, status TEXT)")
db.execute("INSERT INTO auftraege VALUES (4711, 'offen')")
db.commit()

db.execute("UPDATE auftraege SET status = ? WHERE nr = ?", ("fertig", 4711))   # kein commit
andere = sqlite3.connect(pfad)                                                  # zweite Verbindung
print("ohne commit:", andere.execute("SELECT status FROM auftraege").fetchone())

db.commit()
print("mit commit: ", andere.execute("SELECT status FROM auftraege").fetchone())
andere.close()
db.close()

**4. UPDATE und DELETE ohne WHERE treffen alle Zeilen.** Zähle vorher und nachher, und lies `rowcount`:

In [ ]:
db = neue_db()
cur = db.cursor()

cur.execute("UPDATE auftraege SET status = ? WHERE nr = ?", ("fertig", 4713))
print("mit WHERE:  ", cur.rowcount, "Zeile geändert")

cur.execute("UPDATE auftraege SET status = ?", ("fertig",))                      # kein WHERE!
print("ohne WHERE: ", cur.rowcount, "Zeilen geändert")

cur.execute("SELECT COUNT(*) FROM auftraege WHERE status = ?", ("fertig",))
print("fertig sind jetzt", cur.fetchone()[0], "von 12")

**5. Fehlermeldungen lesen.** Die Zelle fängt die typischen Fehler ab und zeigt die Meldung, damit das Notebook weiterläuft. Ordne jede Meldung der Ursache zu (Tabelle, Spalte, Tippfehler im SQL-Wort, doppelter Primärschlüssel, zu wenige Werte):

In [ ]:
db = neue_db()
cur = db.cursor()

versuche = [
    ("SELECT * FROM auftrag", ()),
    ("SELECT nummer FROM auftraege", ()),
    ("SELEC nr FROM auftraege", ()),
    ("INSERT INTO auftraege VALUES (?, ?, ?, ?, ?)", (4711, "ZAPFEN_8", "D2", 100, "offen")),
    ("INSERT INTO auftraege VALUES (?, ?, ?)", (4801, "ZAPFEN_8", "D2")),
]
for sql, werte in versuche:
    try:
        cur.execute(sql, werte)
    except sqlite3.Error as fehler:
        print(type(fehler).__name__ + ":", fehler)

## Stufe 1 — Aufwärmen

### 1.1 Offene Aufträge einer Maschine
Hole die Nummer und die Menge aller Aufträge der Maschine **F1** mit dem Status **offen**. Maschine und Status kommen als **Platzhalter** in die Abfrage (nicht in den Text kleben). Lege das Ergebnis von `fetchall()` in `treffer` ab.

<details><summary>Tipp anzeigen</summary>

`cur.execute("SELECT nr, menge FROM auftraege WHERE maschine = ? AND status = ?", (maschine, status))`, danach `treffer = cur.fetchall()`.
</details>

In [ ]:
db = neue_db()
cur = db.cursor()
maschine = "F1"
status = "offen"

# TODO: Abfrage mit zwei Platzhaltern, Ergebnis in treffer
treffer = None
print(treffer)

In [ ]:
pruefe(sorted(treffer) if treffer else treffer, [(4712, 120), (4716, 200)], "F1 hat zwei offene Aufträge")

<details><summary>Musterlösung anzeigen (erst selbst versuchen)</summary>

```python
cur.execute("SELECT nr, menge FROM auftraege WHERE maschine = ? AND status = ?", (maschine, status))
treffer = cur.fetchall()
```
</details>

### 1.2 Die drei kleinsten Aufträge
Welche drei Aufträge haben die **kleinste Menge**? Gib die Nummern als Liste `kleinste` an, der Auftrag mit der kleinsten Menge zuerst. Sortiere in SQL (`ORDER BY`), begrenze mit `LIMIT`, und baue die Liste aus den Tupeln.

<details><summary>Tipp anzeigen</summary>

`SELECT nr FROM auftraege ORDER BY menge LIMIT 3`. `fetchall()` liefert Tupel wie `(4715,)`, mit `zeile[0]` bekommst du die Zahl.
</details>

In [ ]:
db = neue_db()
cur = db.cursor()

# TODO: ORDER BY menge und LIMIT 3, danach nur die Nummern in eine Liste
kleinste = None
print(kleinste)

In [ ]:
pruefe(kleinste, [4715, 4721, 4712], "Die drei kleinsten Mengen: 60, 90, 120")

<details><summary>Musterlösung anzeigen (erst selbst versuchen)</summary>

```python
cur.execute("SELECT nr FROM auftraege ORDER BY menge LIMIT 3")
kleinste = [zeile[0] for zeile in cur.fetchall()]
```
</details>

### 1.3 Aufträge je Status zählen
Zähle mit **einer** Abfrage, wie viele Aufträge es je Status gibt (`GROUP BY status` mit `COUNT(*)`), und übernimm das Ergebnis in ein Dictionary `je_status` (Status → Anzahl).

<details><summary>Tipp anzeigen</summary>

`SELECT status, COUNT(*) FROM auftraege GROUP BY status`. In der Schleife `for status, anzahl in cur.fetchall():` füllst du das Dictionary: `je_status[status] = anzahl`.
</details>

In [ ]:
db = neue_db()
cur = db.cursor()

# TODO: GROUP BY status, Ergebnis in ein Dictionary übernehmen
je_status = None
print(je_status)

In [ ]:
pruefe(je_status, {"fertig": 4, "in Arbeit": 2, "offen": 6}, "Status zu Anzahl")

<details><summary>Musterlösung anzeigen (erst selbst versuchen)</summary>

```python
cur.execute("SELECT status, COUNT(*) FROM auftraege GROUP BY status")
je_status = {}
for status, anzahl in cur.fetchall():
    je_status[status] = anzahl
```
</details>

## Stufe 2 — Anwenden

### 2.1 Auftrag fertig melden mit Rückmeldung
Schreibe die Funktion `fertig_melden(db, nr)`. Sie setzt den Status des Auftrags `nr` auf `fertig` (nur dieser Auftrag!), schreibt fest und gibt `True` zurück, wenn ein Auftrag geändert wurde, sonst `False` (unbekannte Nummer). Die Anzahl der geänderten Zeilen verrät dir `cur.rowcount`.

<details><summary>Tipp anzeigen</summary>

`cur.execute("UPDATE auftraege SET status = ? WHERE nr = ?", ("fertig", nr))`, dann `db.commit()` und `return cur.rowcount == 1`.
</details>

In [ ]:
def fertig_melden(db, nr):
    # TODO: UPDATE mit WHERE und Platzhaltern, commit, True oder False zurückgeben
    pass


db = neue_db()
print(fertig_melden(db, 4713), fertig_melden(db, 9999))

In [ ]:
db = neue_db()
pruefe(fertig_melden(db, 4713), True, "4713 wurde geändert")
pruefe(fertig_melden(db, 9999), False, "9999 gibt es nicht")
pruefe(db.execute("SELECT COUNT(*) FROM auftraege WHERE status = 'fertig'").fetchone()[0], 5, "Nur ein Auftrag mehr ist fertig (4 + 1)")
pruefe(db.execute("SELECT status FROM auftraege WHERE nr = 4712").fetchone()[0], "offen", "4712 blieb unverändert")

<details><summary>Musterlösung anzeigen (erst selbst versuchen)</summary>

```python
def fertig_melden(db, nr):
    cur = db.cursor()
    cur.execute("UPDATE auftraege SET status = ? WHERE nr = ?", ("fertig", nr))
    db.commit()
    return cur.rowcount == 1
```
</details>

### 2.2 Zwei Tabellen verbinden (JOIN)
Baue für alle Aufträge, die **noch nicht fertig** sind, Zeilen im Format `4711 WELLE_20H6 auf Drehmaschine` (Nummer, Teil, Maschinentyp) als Liste von Texten `zeilen_text`, sortiert nach der Nummer. Den Typ holst du per `JOIN` aus der Tabelle `maschinen`: `auftraege.maschine` verweist auf `maschinen.id`.

<details><summary>Tipp anzeigen</summary>

`SELECT auftraege.nr, auftraege.teil, maschinen.typ FROM auftraege JOIN maschinen ON auftraege.maschine = maschinen.id WHERE auftraege.status != ? ORDER BY auftraege.nr`, Wert `("fertig",)`. Den Text baust du in der Schleife mit einem f-String.
</details>

In [ ]:
db = neue_db()
cur = db.cursor()

# TODO: JOIN, Bedingung mit Platzhalter, Sortierung, dann die Texte bauen
zeilen_text = None
print(zeilen_text)

In [ ]:
pruefe(len(zeilen_text) if zeilen_text else 0, 8, "Acht Aufträge sind noch nicht fertig")
pruefe(zeilen_text[0] if zeilen_text else None, "4711 WELLE_20H6 auf Drehmaschine", "Erste Zeile")
pruefe(zeilen_text[1] if zeilen_text and len(zeilen_text) > 1 else None, "4712 FLANSCH_80 auf Fraesmaschine", "Zweite Zeile")

<details><summary>Musterlösung anzeigen (erst selbst versuchen)</summary>

```python
cur.execute("""SELECT auftraege.nr, auftraege.teil, maschinen.typ
               FROM auftraege
               JOIN maschinen ON auftraege.maschine = maschinen.id
               WHERE auftraege.status != ?
               ORDER BY auftraege.nr""", ("fertig",))
zeilen_text = []
for nr, teil, typ in cur.fetchall():
    zeilen_text.append(f"{nr} {teil} auf {typ}")
```
</details>

### 2.3 Auftragsliste der KW 48 importieren
Die Arbeitsvorbereitung liefert die Aufträge der KW 48 als CSV-Text (Semikolon, mit Kopfzeile, wie `auftraege-kw48.csv` im Labor). Eine Menge ist kaputt („zwölf“). Wandle jede Zeile um, **überspringe kaputte Zeilen** mit `try`/`except ValueError` und merke dir ihre Nummern in `uebersprungen` (Liste von Texten). Füge die gültigen Zeilen mit `executemany` ein und schreibe fest. Danach sollen 16 Aufträge in der Tabelle stehen.

<details><summary>Tipp anzeigen</summary>

Zeilen mit `text.splitlines()`, Kopfzeile mit `[1:]` überspringen, jede Zeile mit `zeile.split(";")` zerlegen. In `try:` die Felder 0 und 3 mit `int(...)` umwandeln. Gültige als Tupel `(nr, teil, maschine, menge, status)` sammeln, dann `executemany` und `commit`.
</details>

In [ ]:
kw48 = """nr;teil;maschine;menge;status
4723;BOLZEN_12;D2;600;offen
4724;FLANSCH_80;F1;80;offen
4725;WELLE_25K6;D2;zwölf;offen
4726;DECKEL_80;F1;150;offen
4727;BUCHSE_30;D2;350;offen"""

db = neue_db()
cur = db.cursor()
gueltig = []
uebersprungen = []

# TODO: Zeilen umwandeln, kaputte überspringen, gültige einfügen, commit
print(gueltig, uebersprungen)

In [ ]:
pruefe(len(gueltig), 4, "Vier gültige Zeilen")
pruefe(uebersprungen, ["4725"], "Die kaputte Zeile 4725 wurde übersprungen")
pruefe(db.execute("SELECT COUNT(*) FROM auftraege").fetchone()[0], 16, "Jetzt stehen 16 Aufträge in der Tabelle")

<details><summary>Musterlösung anzeigen (erst selbst versuchen)</summary>

```python
for text in kw48.splitlines()[1:]:
    felder = text.split(";")
    try:
        nr = int(felder[0])
        menge = int(felder[3])
    except ValueError:
        uebersprungen.append(felder[0])
        continue
    gueltig.append((nr, felder[1], felder[2], menge, felder[4]))

cur.executemany("INSERT INTO auftraege VALUES (?, ?, ?, ?, ?)", gueltig)
db.commit()
```
</details>

## Stufe 3 — Knobeln

### 3.1 Sichere Abfrage als Funktion
Schreibe `auftraege_fuer(cur, maschine, status)`: Sie gibt die **sortierte Liste der Auftragsnummern** zurück, die zu Maschine und Status passen. Die Werte kommen als Platzhalter in die Abfrage. Die Prüfzelle probiert auch `x' OR '1'='1` und einen Namen mit Apostroph aus.

<details><summary>Tipp anzeigen</summary>

`cur.execute("SELECT nr FROM auftraege WHERE maschine = ? AND status = ? ORDER BY nr", (maschine, status))` und `[zeile[0] for zeile in cur.fetchall()]`.
</details>

In [ ]:
def auftraege_fuer(cur, maschine, status):
    # TODO: sortierte Liste der Nummern, mit Platzhaltern
    pass


db = neue_db()
cur = db.cursor()
print(auftraege_fuer(cur, "D2", "offen"))

In [ ]:
pruefe(auftraege_fuer(cur, "D2", "offen"), [4713, 4717, 4719, 4720], "D2 offen")
pruefe(auftraege_fuer(cur, "F1", "in Arbeit"), [4718], "F1 in Arbeit")
pruefe(auftraege_fuer(cur, "x' OR '1'='1", "offen"), [], "Injection-Eingabe liefert keinen Treffer")
cur.execute("INSERT INTO maschinen VALUES (?, ?, ?)", ("O'Brien", "Test", "0"))
cur.execute("INSERT INTO auftraege VALUES (?, ?, ?, ?, ?)", (4900, "ZAPFEN_8", "O'Brien", 10, "offen"))
pruefe(auftraege_fuer(cur, "O'Brien", "offen"), [4900], "Ein Apostroph im Namen bricht nichts")

<details><summary>Musterlösung anzeigen (erst selbst versuchen)</summary>

```python
def auftraege_fuer(cur, maschine, status):
    cur.execute("SELECT nr FROM auftraege WHERE maschine = ? AND status = ? ORDER BY nr", (maschine, status))
    return [zeile[0] for zeile in cur.fetchall()]
```
</details>

### 3.2 Teilsuche mit LIKE: das Prozentzeichen gehört in den Wert
Mit `LIKE` sucht SQL nach Mustern: `%` steht für beliebig viele Zeichen. Schreibe `teile_mit(cur, stueck)`: Sie gibt die **verschiedenen Teilenamen** (sortiert, ohne Doppelte) zurück, die den Text `stueck` enthalten, zum Beispiel `teile_mit(cur, "WELLE")`. KI-Code schreibt hier gern `LIKE '%?%'`: Das ist falsch, denn ein `?` zwischen Anführungszeichen ist kein Platzhalter mehr, sondern ein Fragezeichen im Text. Das `%` muss **in den Wert**: `("%" + stueck + "%",)`.

<details><summary>Tipp anzeigen</summary>

`SELECT DISTINCT teil FROM auftraege WHERE teil LIKE ? ORDER BY teil` mit dem Wert `("%" + stueck + "%",)`. `DISTINCT` lässt Doppelte weg.
</details>

In [ ]:
def teile_mit(cur, stueck):
    # TODO: LIKE mit Platzhalter; das Prozentzeichen steht im Wert, nicht im SQL-Text
    pass


db = neue_db()
cur = db.cursor()
print(teile_mit(cur, "WELLE"))

In [ ]:
pruefe(teile_mit(cur, "WELLE"), ["WELLE_20H6", "WELLE_25K6"], "Teile mit WELLE")
pruefe(teile_mit(cur, "_80"), ["DECKEL_80", "FLANSCH_80"], "Teile mit _80")
pruefe(teile_mit(cur, "gibt_es_nicht"), [], "Kein Treffer: leere Liste")
pruefe(teile_mit(cur, "x' OR '1'='1"), [], "Injection-Eingabe liefert keinen Treffer")

<details><summary>Musterlösung anzeigen (erst selbst versuchen)</summary>

```python
def teile_mit(cur, stueck):
    cur.execute("SELECT DISTINCT teil FROM auftraege WHERE teil LIKE ? ORDER BY teil", ("%" + stueck + "%",))
    return [zeile[0] for zeile in cur.fetchall()]
```
</details>

### 3.3 Beides oder nichts: Material buchen und Auftrag freigeben
Eine Auftragsfreigabe besteht aus **zwei Änderungen**: Der Status des Auftrags wird `in Arbeit`, und der Lagerbestand des Teils sinkt um die Menge. Gibt es zu wenig Material, darf **keine** der beiden Änderungen gelten, sonst steht ein Auftrag in Arbeit, für den das Material fehlt. Das nennt man eine **Transaktion**: alles oder nichts. Die Tabelle `lager` hat eine Regel `CHECK (bestand >= 0)`: Ein zu kleiner Bestand löst einen `sqlite3.IntegrityError` aus. Schreibe `freigeben(db, nr, teil, menge)`: beide UPDATEs stehen in einem `with db:`-Block (der Block schreibt bei Erfolg fest und macht bei einem Fehler **alles** rückgängig), den Fehler fängst du **außerhalb** des Blocks ab. Rückgabe `True` bei Erfolg, `False` bei zu wenig Material.

<details><summary>Tipp anzeigen</summary>

```python
with db:
    db.execute("UPDATE auftraege SET status = ? WHERE nr = ?", ("in Arbeit", nr))
    db.execute("UPDATE lager SET bestand = bestand - ? WHERE teil = ?", (menge, teil))
```
Umschließe diesen Block mit `try:` und `except sqlite3.IntegrityError:` (`return False`). Das `with db:` macht bei dem Fehler auch das erste UPDATE rückgängig.
</details>

In [ ]:
def lager_db():
    db = neue_db()
    db.execute("CREATE TABLE lager (teil TEXT PRIMARY KEY, bestand INTEGER CHECK (bestand >= 0))")
    db.executemany("INSERT INTO lager VALUES (?, ?)", [("WELLE_20H6", 400), ("FLANSCH_80", 100)])
    db.commit()
    return db


def freigeben(db, nr, teil, menge):
    # TODO: zwei UPDATEs in einem with-db-Block, IntegrityError außerhalb abfangen
    pass


db = lager_db()
print(freigeben(db, 4713, "WELLE_20H6", 300), freigeben(db, 4712, "FLANSCH_80", 120))

In [ ]:
db = lager_db()
pruefe(freigeben(db, 4713, "WELLE_20H6", 300), True, "4713: genug Material, freigegeben")
pruefe(db.execute("SELECT status FROM auftraege WHERE nr = 4713").fetchone()[0], "in Arbeit", "4713 steht auf in Arbeit")
pruefe(db.execute("SELECT bestand FROM lager WHERE teil = 'WELLE_20H6'").fetchone()[0], 100, "Bestand WELLE_20H6: 400 - 300")
pruefe(freigeben(db, 4712, "FLANSCH_80", 120), False, "4712: zu wenig Material (100 < 120), abgelehnt")
pruefe(db.execute("SELECT status FROM auftraege WHERE nr = 4712").fetchone()[0], "offen", "4712 blieb offen: der Status wurde zurückgerollt")
pruefe(db.execute("SELECT bestand FROM lager WHERE teil = 'FLANSCH_80'").fetchone()[0], 100, "Bestand FLANSCH_80 unverändert")

<details><summary>Musterlösung anzeigen (erst selbst versuchen)</summary>

```python
def lager_db():
    db = neue_db()
    db.execute("CREATE TABLE lager (teil TEXT PRIMARY KEY, bestand INTEGER CHECK (bestand >= 0))")
    db.executemany("INSERT INTO lager VALUES (?, ?)", [("WELLE_20H6", 400), ("FLANSCH_80", 100)])
    db.commit()
    return db


def freigeben(db, nr, teil, menge):
    try:
        with db:
            db.execute("UPDATE auftraege SET status = ? WHERE nr = ?", ("in Arbeit", nr))
            db.execute("UPDATE lager SET bestand = bestand - ? WHERE teil = ?", (menge, teil))
    except sqlite3.IntegrityError:
        return False
    return True
```
</details>

### 3.4 Code-Review: Würdest du das freigeben? (frei)
Eine KI hat diese Funktion für die Zelle geschrieben. Sie soll alle **fertigen** Aufträge einer Maschine löschen und die Anzahl zurückgeben. Finde die Fehler, **bevor** du die Lösung aufklappst. Es sind zwei, und einer davon ist gefährlich.

```python
def fertige_loeschen(db, maschine):
    cur = db.cursor()
    cur.execute(f"DELETE FROM auftraege WHERE maschine = '{maschine}' AND status = 'fertig'")
    return cur.rowcount
```

<details><summary>Tipp anzeigen</summary>

Schau auf die Werte im SQL-Text (Injection, Apostroph) und auf das, was nach dem DELETE fehlt. Überlege, was die Eingabe `' OR 1=1 --` aus dem SQL-Text macht.
</details>

<details><summary>Lösung anzeigen</summary>

1. **f-String im SQL-Text:** Mit der Eingabe `' OR 1=1 --` wird der Rest der Zeile zum SQL-Kommentar (`--`), und die Bedingung ist für jede Zeile wahr: Es wird die **ganze Tabelle** gelöscht. Mit einem Apostroph im Namen bricht die Abfrage ab. Lösung: `WHERE maschine = ? AND status = ?` mit `(maschine, "fertig")`.
2. **Kein `commit()`:** Das Löschen bleibt in einer offenen Transaktion und ist bei einer Datei-Datenbank nach `close()` weg. Am elegantesten schreibst du das DELETE in einen `with db:`-Block, der bei Erfolg festschreibt und bei einem Fehler zurückrollt.

Korrigiert:

```python
def fertige_loeschen(db, maschine):
    with db:
        cur = db.execute("DELETE FROM auftraege WHERE maschine = ? AND status = ?", (maschine, "fertig"))
    return cur.rowcount
```
</details>

In [ ]:
# Zum Ausprobieren: Deine korrigierte Fassung hier
def fertige_loeschen(db, maschine):
    pass


db = neue_db()
print(fertige_loeschen(db, "F1"))      # F1 hat zwei fertige Aufträge (4715, 4721)

In [ ]:
db = neue_db()
pruefe(fertige_loeschen(db, "F1"), 2, "F1: zwei fertige Aufträge gelöscht")
pruefe(db.execute("SELECT COUNT(*) FROM auftraege").fetchone()[0], 10, "Zehn Aufträge bleiben")
db = neue_db()
pruefe(fertige_loeschen(db, "x' OR '1'='1"), 0, "Injection-Eingabe löscht nichts")
pruefe(db.execute("SELECT COUNT(*) FROM auftraege").fetchone()[0], 12, "Alle zwölf Aufträge sind noch da")

## Ausblick: pandas, Server-Datenbanken (freiwillig)
**pandas liest SQL direkt** (Brücke zu V15): Mit `pd.read_sql_query("SELECT ...", db)` wird das Ergebnis einer Abfrage ein DataFrame, auf dem du `groupby`, Diagramme und alles aus V15 und V14 anwenden kannst. Bei Platzhaltern übergibst du die Werte mit `params=(...)`.

```python
import pandas as pd

df = pd.read_sql_query("SELECT maschine, menge FROM auftraege WHERE status != ?", db, params=("fertig",))
print(df.groupby("maschine")["menge"].sum())
```

**Was hier nicht vorkam, du aber kennen solltest:** SQLite ist eine Datenbank in einer Datei und ideal für kleine Werkzeuge, Smartphones und eingebettete Geräte. Die Auftragsverwaltung eines Werks läuft dagegen auf einer **Server-Datenbank** (PostgreSQL, SQL Server, Oracle) hinter einem MES oder ERP-System. Der Python-Code ändert sich dabei kaum: andere `connect`-Zeile, dasselbe SQL, **dieselben Platzhalter** (je nach Bibliothek `?` oder `%s`). Wie man ein großes Datenmodell entwirft (Normalformen) und Abfragen mit Indizes beschleunigt, ist eine Aufgabe für Spezialisten: Du musst das Ergebnis lesen und mit ihnen sprechen können.

## Selbsttest: Kannst du es im Kopf?
Beantworte erst auf Papier, dann prüfe in der Zelle darunter.

1. Welche Spalte ist der Primärschlüssel von `auftraege`, und was passiert, wenn du einen zweiten Auftrag mit Nummer 4711 einfügst?
2. Was liefert `SELECT maschine, COUNT(*) FROM auftraege GROUP BY maschine ORDER BY maschine`?
3. Was ist der Unterschied zwischen `("D2")` und `("D2",)`?
4. Warum ist `cur.execute(f"SELECT nr FROM auftraege WHERE teil = '{eingabe}'")` gefährlich, und wie sieht die sichere Fassung aus?
5. Was geht verloren, wenn du bei einer Datei-Datenbank nach einem UPDATE das `commit()` vergisst?

In [ ]:
db = neue_db()
cur = db.cursor()

try:
    cur.execute("INSERT INTO auftraege VALUES (?, ?, ?, ?, ?)", (4711, "ZAPFEN_8", "D2", 100, "offen"))   # 1
except sqlite3.IntegrityError as fehler:
    print(fehler)

cur.execute("SELECT maschine, COUNT(*) FROM auftraege GROUP BY maschine ORDER BY maschine")           # 2
print(cur.fetchall())
print(type(("D2")), type(("D2",)))                                                                    # 3
eingabe = "x' OR '1'='1"                                                                              # 4
cur.execute("SELECT nr FROM auftraege WHERE teil = ?", (eingabe,))
print(len(cur.fetchall()), "Treffer mit Platzhalter")

## Eigene Frage an die Daten (frei)
Wähle selbst: Welche Maschine hat die größte Gesamtmenge noch nicht fertiger Aufträge? Welches Teil kommt in den meisten Aufträgen vor? Welcher Auftrag hat die größte Menge je Maschine? Formuliere die Frage in SQL (`GROUP BY`, `ORDER BY`, `LIMIT`), schreibe sie in die Zelle und beantworte sie mit einem `print`. Es gibt keine Prüfzelle. Erkläre einem Mitstudierenden, ob deine Abfrage Platzhalter braucht, und wenn ja, wo das Tupel steht.

In [ ]:
db = neue_db()
cur = db.cursor()

# Dein Code